# RAPID2 — Production Data EDA

## Objective

This notebook establishes a **production-safe, BigQuery-first EDA baseline** for RAPID2, the live regulatory-record source used by the Reg Management AI development team.

The EDA is designed to answer the questions that matter before AI solution design:

- What is the population and grain?
- How much data is available and how fresh is it?
- How complete and unique are the important fields?
- What are the major jurisdictions, statuses, categories, regulators and ingest channels?
- How does the population change over time?
- Are there duplicates or logical inconsistencies?
- Is the regulatory text suitable for AI / NLP?
- How well are themes and risk taxonomy populated?
- How much of the population is linked to alerts?
- Can the source support traceable, time-aware AI answers?

### Design principle

RAPID2 is live PROD data. Therefore, **do not pull the full table into pandas**. Large aggregations stay in BigQuery; pandas receives only compact summaries and controlled samples.

The notebook also keeps the current RAPID2 extraction logic conceptually separate from the AI logic so that the future strategic solution can replace the tactical access layer without rewriting the EDA methodology.

## 1. Configuration

The values below follow the RAPID2 extraction notebook shared by the team. The jurisdiction is deliberately parameterised because the source notebook notes that data-visa approval currently limits the approved jurisdictions.

**Change only this section when the approved jurisdiction or environment changes.**

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import bigquery

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------

ANALYTICS_PROJECT = "hsbc-12211209-cmlpwb-prod"
DATA_PROJECT = "hsbc-11545401-cmpdwrs1-prod"
DATASET = "rc_curated_prod"
BASE_TABLE = "aa_hsbc_rapid2_hzn_record"

# Reference / enrichment tables used by the team's RAPID2 extraction.
REG_REF_TABLE = "aa_hsbc_rapid2_reg_ref_jris"
STAT_REF_TABLE = "aa_hsbc_rapid2_hzn_ref_stat"
ALERT_TABLE = "aa_hsbc_rapid2_hzn_alert"
THEME_MAP_TABLE = "aa_hsbc_rapid2_hzn_record_theme_map"
THEME_REF_TABLE = "aa_hsbc_rapid2_hzn_ref_theme"
TAXONOMY_TABLE = "aa_hsbc_rapid2_hzn_record_risk_txnmy"

LOCATION = "europe-west2"

# Data-visa / business filter. Keep this explicit in every production query.
RAPID2_JURISDICTION = "Hong Kong (HSBC)"

# Optional scope filters. Set to None to disable.
CREATED_FROM = "2020-01-01"
CREATED_TO = "2025-12-31"

# The source notebook shows these as optional filters. Leave False unless
# the EDA scope is specifically intended to reproduce that subset.
FILTER_ACTIVE_ONLY = False
FILTER_INGEST_CHANNEL = False
INGEST_CHANNELS = (2, 8)
FILTER_RECORD_CATEGORY = False
RECORD_CATEGORY = "REGREV"

TOP_N = 20
SAMPLE_N = 1000

os.environ["GOOGLE_CLOUD_PROJECT"] = ANALYTICS_PROJECT
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/home/jovyan/.config/gcloud/application_default_credentials.json"

BQ = bigquery.Client(project=ANALYTICS_PROJECT, location=LOCATION)

TABLE = f"`{DATA_PROJECT}.{DATASET}.{BASE_TABLE}`"
DATASET_REF = f"`{DATA_PROJECT}.{DATASET}`"

print("RAPID2 base table:", TABLE)
print("Jurisdiction:", RAPID2_JURISDICTION)

## 2. Reusable query helpers

The notebook uses small helper functions to keep the EDA readable and minimise repeated boilerplate.

In [ ]:
def run_query(sql, label=None):
    """Run BigQuery SQL and return a pandas DataFrame."""
    job_config = bigquery.QueryJobConfig()
    if label:
        job_config.labels = {"eda": label[:63].lower().replace("_", "-")}
    return BQ.query(sql, job_config=job_config).result().to_dataframe(
        create_bqstorage_client=False
    )


def pct(numerator, denominator):
    return round(100 * numerator / denominator, 2) if denominator else 0

print("Helpers loaded.")

## 3. Establish the EDA population

The team notebook joins RAPID2 records to reference data. For the EDA, we define the **base population once** and reuse the same scope logic throughout.

This is important: otherwise different EDA cells can accidentally profile different populations.

The default scope uses:

- the live RAPID2 production table;
- the approved jurisdiction;
- the date window shown in the team notebook;
- optional active / channel / category filters controlled above.

In [ ]:
where_clauses = [f'JRIS_CDE = "{RAPID2_JURISDICTION}"']

if CREATED_FROM:
    where_clauses.append(f'DATE(CREATED_ON_DTM) >= "{CREATED_FROM}"')
if CREATED_TO:
    where_clauses.append(f'DATE(CREATED_ON_DTM) <= "{CREATED_TO}"')
if FILTER_ACTIVE_ONLY:
    where_clauses.append('IS_ACTV_IND = 1')
if FILTER_INGEST_CHANNEL:
    channels = ", ".join(map(str, INGEST_CHANNELS))
    where_clauses.append(f'INGEST_CHANL_ID IN ({channels})')
if FILTER_RECORD_CATEGORY:
    where_clauses.append(f'RECORD_CAT_CDE = "{RECORD_CATEGORY}"')

WHERE_SQL = "\n    AND ".join(where_clauses)

print(WHERE_SQL)

## 4. Source metadata and physical schema

Before looking at values, establish the physical source: table size, modification time, partitioning/clustering information and column definitions.

In [ ]:
metadata_sql = f"""
SELECT
    TABLE_CATALOG AS PROJECT_ID,
    TABLE_SCHEMA AS DATASET_ID,
    TABLE_NAME,
    TABLE_TYPE,
    CREATION_TIME,
    DDL
FROM {DATASET_REF}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_NAME = "{BASE_TABLE}"
"""

metadata_df = run_query(metadata_sql, "metadata")
display(metadata_df)

size_sql = f"""
SELECT
    TABLE_ID AS TABLE_NAME,
    ROW_COUNT,
    SIZE_BYTES,
    ROUND(SIZE_BYTES / POW(1024, 3), 2) AS SIZE_GB,
    TIMESTAMP_MILLIS(LAST_MODIFIED_TIME) AS LAST_MODIFIED_AT
FROM {DATA_PROJECT}.{DATASET}.__TABLES__
WHERE TABLE_ID = "{BASE_TABLE}"
"""

display(run_query(size_sql, "table_size"))

schema_sql = f"""
SELECT
    ORDINAL_POSITION,
    COLUMN_NAME,
    DATA_TYPE,
    IS_NULLABLE,
    DESCRIPTION
FROM {DATASET_REF}.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = "{BASE_TABLE}"
ORDER BY ORDINAL_POSITION
"""

schema_df = run_query(schema_sql, "schema")
display(schema_df)
print(f"Physical columns: {len(schema_df)}")

## 5. Core population profile

This is the first executive-level EDA output. It establishes the population, date coverage, active/inactive mix, alert linkage and content availability.

The query is intentionally one aggregation rather than multiple scans.

In [ ]:
core_sql = f"""
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_ID,
    COUNT(DISTINCT ORIGIN_ALERT_ID) AS DISTINCT_ALERT_ID,

    MIN(CREATED_ON_DTM) AS MIN_CREATED_ON_DTM,
    MAX(CREATED_ON_DTM) AS MAX_CREATED_ON_DTM,
    MIN(UPDATED_ON_DTM) AS MIN_UPDATED_ON_DTM,
    MAX(UPDATED_ON_DTM) AS MAX_UPDATED_ON_DTM,

    COUNTIF(IS_ACTV_IND = 1) AS ACTIVE_RECORDS,
    COUNTIF(IS_ACTV_IND = 0) AS INACTIVE_RECORDS,
    COUNTIF(ORIGIN_ALERT_ID IS NOT NULL) AS ALERT_LINKED_RECORDS,
    COUNTIF(ORIGIN_ALERT_ID IS NULL) AS NON_ALERT_LINKED_RECORDS,

    COUNTIF(TITLE IS NOT NULL AND TRIM(TITLE) != "") AS TITLE_AVAILABLE,
    COUNTIF(SUMM_CONTNT IS NOT NULL AND TRIM(SUMM_CONTNT) != "") AS SUMMARY_AVAILABLE,
    COUNTIF(INTRO IS NOT NULL AND TRIM(INTRO) != "") AS INTRO_AVAILABLE
FROM {TABLE}
WHERE {WHERE_SQL}
"""

core_df = run_query(core_sql, "core_profile")
display(core_df.T)

## 6. Field completeness and cardinality

For high-value fields we calculate:

- null count / null percentage;
- approximate distinct count;
- distinct percentage.

This identifies candidate keys, low-cardinality dimensions and sparse fields.

`APPROX_COUNT_DISTINCT` keeps this scalable on production data.

In [ ]:
PROFILE_COLUMNS = [
    "RECORD_ID", "ORIGIN_ALERT_ID", "INGEST_RULE_TYPE", "INGEST_CHANL_ID",
    "JRIS_CDE", "CREATED_ON_DTM", "UPDATED_ON_DTM", "IS_ACTV_IND",
    "RECORD_CAT_CDE", "STAT_CDE", "RGLT_BDY_CDE", "RISK_STWRD_AREA_CDE",
    "TITLE", "SUMM_CONTNT", "SUMM_UPDT", "SCR_URL", "INTRO"
]

available = set(schema_df["COLUMN_NAME"].str.upper())
PROFILE_COLUMNS = [c for c in PROFILE_COLUMNS if c in available]

parts = []
for col in PROFILE_COLUMNS:
    parts.append(f"""
SELECT
    '{col}' AS COLUMN_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNTIF(`{col}` IS NULL) AS NULL_COUNT,
    ROUND(100 * SAFE_DIVIDE(COUNTIF(`{col}` IS NULL), COUNT(*)), 2) AS NULL_PCT,
    APPROX_COUNT_DISTINCT(`{col}`) AS APPROX_DISTINCT
FROM {TABLE}
WHERE {WHERE_SQL}
""")

completeness_df = run_query("\nUNION ALL\n".join(parts), "completeness")
completeness_df = completeness_df.sort_values("NULL_PCT", ascending=False)
display(completeness_df)

## 7. Key / duplicate analysis

A stable `RECORD_ID` is fundamental for AI traceability. We also check exact repetition of title, summary and source URL.

Potential duplicate groups are surfaced separately because repeated titles do not necessarily mean duplicate regulatory records.

In [ ]:
duplicate_sql = f"""
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_ID,
    COUNT(*) - COUNT(DISTINCT RECORD_ID) AS DUPLICATE_RECORD_ID_ROWS,
    COUNTIF(RECORD_ID IS NULL) AS NULL_RECORD_ID,
    COUNT(DISTINCT TITLE) AS DISTINCT_TITLE,
    COUNT(DISTINCT SUMM_CONTNT) AS DISTINCT_SUMMARY,
    COUNT(DISTINCT SCR_URL) AS DISTINCT_URL
FROM {TABLE}
WHERE {WHERE_SQL}
"""

display(run_query(duplicate_sql, "duplicates").T)

potential_dup_sql = f"""
SELECT
    JRIS_CDE,
    RGLT_BDY_CDE,
    TITLE,
    COUNT(*) AS RECORD_COUNT,
    MIN(CREATED_ON_DTM) AS FIRST_CREATED,
    MAX(CREATED_ON_DTM) AS LAST_CREATED
FROM {TABLE}
WHERE {WHERE_SQL}
  AND TITLE IS NOT NULL
  AND TRIM(TITLE) != ""
GROUP BY JRIS_CDE, RGLT_BDY_CDE, TITLE
HAVING COUNT(*) > 1
ORDER BY RECORD_COUNT DESC
LIMIT {TOP_N}
"""

display(run_query(potential_dup_sql, "potential_duplicates"))

## 8. Important categorical distributions

The initial data landscape should show where the population is concentrated.

Dimensions:

- jurisdiction;
- ingest rule type;
- ingest channel;
- record category;
- status;
- regulator;
- risk steward area.

In [ ]:
CATEGORY_FIELDS = {
    "JURISDICTION": "JRIS_CDE",
    "INGEST_RULE_TYPE": "INGEST_RULE_TYPE",
    "INGEST_CHANNEL": "INGEST_CHANL_ID",
    "RECORD_CATEGORY": "RECORD_CAT_CDE",
    "STATUS": "STAT_CDE",
    "REGULATOR": "RGLT_BDY_CDE",
    "RISK_STEWARD_AREA": "RISK_STWRD_AREA_CDE",
}

available = set(schema_df["COLUMN_NAME"].str.upper())


def top_values(column, label, n=TOP_N):
    if column not in available:
        print(f"{column}: not present in physical table")
        return None
    sql = f"""
    SELECT
        `{column}` AS VALUE,
        COUNT(*) AS RECORD_COUNT,
        ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER()), 2) AS PCT_OF_RECORDS
    FROM {TABLE}
    WHERE {WHERE_SQL}
    GROUP BY `{column}`
    ORDER BY RECORD_COUNT DESC
    LIMIT {n}
    """
    df = run_query(sql, f"top_{label.lower()}")
    print(f"\n### {label}")
    display(df)
    return df

category_results = {
    label: top_values(column, label)
    for label, column in CATEGORY_FIELDS.items()
}

## 9. Lifecycle and status analysis

Regulatory records evolve. We compare status and active indicator, and quantify records that have been updated after creation.

In [ ]:
lifecycle_sql = f"""
SELECT
    STAT_CDE AS STATUS,
    IS_ACTV_IND AS ACTIVE_INDICATOR,
    COUNT(*) AS RECORD_COUNT,
    MIN(CREATED_ON_DTM) AS FIRST_CREATED,
    MAX(CREATED_ON_DTM) AS LAST_CREATED,
    MIN(UPDATED_ON_DTM) AS FIRST_UPDATED,
    MAX(UPDATED_ON_DTM) AS LAST_UPDATED
FROM {TABLE}
WHERE {WHERE_SQL}
GROUP BY STATUS, ACTIVE_INDICATOR
ORDER BY RECORD_COUNT DESC
"""

display(run_query(lifecycle_sql, "lifecycle"))

## 10. Monthly volume and freshness trend

Monthly creation volume reveals ingestion spikes, backfills and periods of inactivity. We also calculate update lag percentiles.

In [ ]:
monthly_sql = f"""
SELECT
    DATE_TRUNC(DATE(CREATED_ON_DTM), MONTH) AS MONTH,
    COUNT(*) AS RECORD_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORDS,
    COUNTIF(ORIGIN_ALERT_ID IS NOT NULL) AS ALERT_LINKED_RECORDS,
    COUNTIF(IS_ACTV_IND = 1) AS ACTIVE_RECORDS
FROM {TABLE}
WHERE {WHERE_SQL}
  AND CREATED_ON_DTM IS NOT NULL
GROUP BY MONTH
ORDER BY MONTH
"""

monthly_df = run_query(monthly_sql, "monthly_volume")
display(monthly_df.tail(30))

plt.figure(figsize=(14, 5))
plt.plot(monthly_df["MONTH"], monthly_df["RECORD_COUNT"])
plt.title("RAPID2 — Monthly Record Creation Volume")
plt.xlabel("Month")
plt.ylabel("Records Created")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

latency_sql = f"""
SELECT
    APPROX_QUANTILES(TIMESTAMP_DIFF(UPDATED_ON_DTM, CREATED_ON_DTM, DAY), 100)[OFFSET(50)] AS P50_LAG_DAYS,
    APPROX_QUANTILES(TIMESTAMP_DIFF(UPDATED_ON_DTM, CREATED_ON_DTM, DAY), 100)[OFFSET(90)] AS P90_LAG_DAYS,
    APPROX_QUANTILES(TIMESTAMP_DIFF(UPDATED_ON_DTM, CREATED_ON_DTM, DAY), 100)[OFFSET(95)] AS P95_LAG_DAYS,
    APPROX_QUANTILES(TIMESTAMP_DIFF(UPDATED_ON_DTM, CREATED_ON_DTM, DAY), 100)[OFFSET(99)] AS P99_LAG_DAYS,
    COUNTIF(UPDATED_ON_DTM < CREATED_ON_DTM) AS UPDATE_BEFORE_CREATE
FROM {TABLE}
WHERE {WHERE_SQL}
  AND CREATED_ON_DTM IS NOT NULL
  AND UPDATED_ON_DTM IS NOT NULL
"""

display(run_query(latency_sql, "update_latency").T)

## 11. Text EDA — AI / NLP readiness

This is deliberately deeper than a simple null check.

We measure:

- title / summary / intro availability;
- character-length percentiles;
- short-content rates;
- long-content rates.

This gives an early indication of whether RAPID2 is suitable for search, RAG, summarisation or classification.

In [ ]:
text_sql = f"""
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNTIF(TITLE IS NULL OR TRIM(TITLE) = "") AS EMPTY_TITLE,
    COUNTIF(SUMM_CONTNT IS NULL OR TRIM(SUMM_CONTNT) = "") AS EMPTY_SUMMARY,
    COUNTIF(INTRO IS NULL OR TRIM(INTRO) = "") AS EMPTY_INTRO,

    APPROX_QUANTILES(LENGTH(TRIM(TITLE)), 100)[OFFSET(50)] AS TITLE_P50_CHARS,
    APPROX_QUANTILES(LENGTH(TRIM(SUMM_CONTNT)), 100)[OFFSET(50)] AS SUMMARY_P50_CHARS,
    APPROX_QUANTILES(LENGTH(TRIM(SUMM_CONTNT)), 100)[OFFSET(90)] AS SUMMARY_P90_CHARS,
    APPROX_QUANTILES(LENGTH(TRIM(SUMM_CONTNT)), 100)[OFFSET(99)] AS SUMMARY_P99_CHARS,

    COUNTIF(LENGTH(TRIM(SUMM_CONTNT)) < 100) AS SUMMARY_LT_100_CHARS,
    COUNTIF(LENGTH(TRIM(SUMM_CONTNT)) >= 1000) AS SUMMARY_GE_1000_CHARS
FROM {TABLE}
WHERE {WHERE_SQL}
"""

display(run_query(text_sql, "text_profile").T)

## 12. Content availability buckets

For AI design, the question is not only “is the field null?” but “does a record contain enough usable context?”

In [ ]:
content_sql = f"""
SELECT
    CASE
        WHEN TITLE IS NULL OR TRIM(TITLE) = "" THEN "NO_TITLE"
        WHEN SUMM_CONTNT IS NULL OR TRIM(SUMM_CONTNT) = "" THEN "TITLE_ONLY"
        WHEN LENGTH(TRIM(SUMM_CONTNT)) < 100 THEN "SHORT_SUMMARY"
        WHEN LENGTH(TRIM(SUMM_CONTNT)) < 1000 THEN "MEDIUM_SUMMARY"
        ELSE "LONG_SUMMARY"
    END AS CONTENT_BUCKET,
    COUNT(*) AS RECORD_COUNT,
    ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER()), 2) AS PCT_OF_RECORDS
FROM {TABLE}
WHERE {WHERE_SQL}
GROUP BY CONTENT_BUCKET
ORDER BY RECORD_COUNT DESC
"""

display(run_query(content_sql, "content_buckets"))

## 13. Text duplication / boilerplate screening

Exact duplicate content can bias retrieval and model evaluation. This first pass identifies repeated summaries and titles.

For a later iteration, use normalised text hashes and, if justified by the use case, MinHash / embedding similarity for near-duplicate detection.

In [ ]:
boilerplate_sql = f"""
SELECT
    SUMM_CONTNT,
    COUNT(*) AS RECORD_COUNT
FROM {TABLE}
WHERE {WHERE_SQL}
  AND SUMM_CONTNT IS NOT NULL
  AND TRIM(SUMM_CONTNT) != ""
GROUP BY SUMM_CONTNT
HAVING COUNT(*) > 1
ORDER BY RECORD_COUNT DESC
LIMIT {TOP_N}
"""

display(run_query(boilerplate_sql, "repeated_summary"))

## 14. Theme and risk-taxonomy coverage

The team's extraction enriches RAPID2 with themes and L1/L2/L3 risk taxonomy using separate reference tables. Those relationships should be profiled separately from the base record grain.

The following section first checks whether the fields already exist physically in the base table. If not, the next section uses the same joins as the team's notebook.

In [ ]:
base_columns = set(schema_df["COLUMN_NAME"].str.upper())

for col in ["RISK_TXNMY_LVL_1", "RISK_TXNMY_LVL_2", "RISK_TXNMY_LVL_3", "RECORD_THEME_CODES"]:
    print(f"{col}: {'present' if col in base_columns else 'not present in base table'}")

## 15. RAPID2 enrichment joins — themes and risk taxonomy

This reproduces the important enrichment pattern from the team's notebook without materialising the full enriched dataset.

### Why this matters

The AI solution will eventually need to reason over regulatory records plus their metadata. We therefore need to know whether the enrichment is:

- complete;
- one-to-one or one-to-many;
- capable of multiplying records;
- leaving records unmatched.

**Do not assume join cardinality from the notebook code. Measure it.**

In [ ]:
# Theme join integrity

theme_join_sql = f"""
WITH BASE AS (
    SELECT RECORD_ID
    FROM {TABLE}
    WHERE {WHERE_SQL}
),
THEME_MAP AS (
    SELECT RECORD_ID, THEME_CODE
    FROM `{DATA_PROJECT}.{DATASET}.{THEME_MAP_TABLE}`
),
THEME_COUNTS AS (
    SELECT
        RECORD_ID,
        COUNT(*) AS THEME_LINK_COUNT
    FROM THEME_MAP
    GROUP BY RECORD_ID
)
SELECT
    COUNT(*) AS BASE_RECORDS,
    COUNTIF(THEME_COUNTS.RECORD_ID IS NULL) AS NO_THEME_LINK,
    COUNTIF(THEME_LINK_COUNT = 1) AS ONE_THEME_LINK,
    COUNTIF(THEME_LINK_COUNT > 1) AS MULTIPLE_THEME_LINKS,
    MAX(COALESCE(THEME_LINK_COUNT, 0)) AS MAX_THEME_LINKS_PER_RECORD
FROM BASE
LEFT JOIN THEME_COUNTS USING (RECORD_ID)
"""

display(run_query(theme_join_sql, "theme_join_integrity").T)

# Risk taxonomy join integrity

taxonomy_join_sql = f"""
WITH BASE AS (
    SELECT RECORD_ID
    FROM {TABLE}
    WHERE {WHERE_SQL}
),
TAXONOMY AS (
    SELECT RECORD_ID, RISK_TXNMY_LVL_1, RISK_TXNMY_LVL_2, RISK_TXNMY_LVL_3
    FROM `{DATA_PROJECT}.{DATASET}.{TAXONOMY_TABLE}`
),
TAX_COUNTS AS (
    SELECT
        RECORD_ID,
        COUNT(*) AS TAXONOMY_LINK_COUNT,
        COUNTIF(RISK_TXNMY_LVL_1 IS NOT NULL) AS L1_PRESENT,
        COUNTIF(RISK_TXNMY_LVL_2 IS NOT NULL) AS L2_PRESENT,
        COUNTIF(RISK_TXNMY_LVL_3 IS NOT NULL) AS L3_PRESENT
    FROM TAXONOMY
    GROUP BY RECORD_ID
)
SELECT
    COUNT(*) AS BASE_RECORDS,
    COUNTIF(TAX_COUNTS.RECORD_ID IS NULL) AS NO_TAXONOMY_LINK,
    COUNTIF(TAXONOMY_LINK_COUNT = 1) AS ONE_TAXONOMY_LINK,
    COUNTIF(TAXONOMY_LINK_COUNT > 1) AS MULTIPLE_TAXONOMY_LINKS,
    MAX(COALESCE(TAXONOMY_LINK_COUNT, 0)) AS MAX_TAXONOMY_LINKS_PER_RECORD
FROM BASE
LEFT JOIN TAX_COUNTS USING (RECORD_ID)
"""

display(run_query(taxonomy_join_sql, "taxonomy_join_integrity").T)

## 16. Alert linkage

The production RAPID2 query exposes the source alert relationship as `ORIGIN_ALERT_ID`.

This analysis establishes the proportion of regulatory records with and without an alert linkage. If alert information is later used as a label or signal, the relationship must be validated with the alert data owner first.

In [ ]:
alert_sql = f"""
SELECT
    CASE WHEN ORIGIN_ALERT_ID IS NULL THEN "NO_ALERT_LINK" ELSE "ALERT_LINKED" END AS ALERT_LINK_STATUS,
    COUNT(*) AS RECORD_COUNT,
    ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER()), 2) AS PCT_OF_RECORDS
FROM {TABLE}
WHERE {WHERE_SQL}
GROUP BY ALERT_LINK_STATUS
ORDER BY RECORD_COUNT DESC
"""

display(run_query(alert_sql, "alert_linkage"))

## 17. Source URL quality

The source URL is important for traceability and human verification. This is a **format check only**; it does not make outbound HTTP requests.

In [ ]:
if "SCR_URL" in base_columns:
    url_sql = f"""
    SELECT
        COUNT(*) AS TOTAL_ROWS,
        COUNTIF(SCR_URL IS NULL OR TRIM(SCR_URL) = "") AS MISSING_URL,
        COUNTIF(REGEXP_CONTAINS(TRIM(SCR_URL), r"^https?://")) AS HTTP_URLS,
        COUNTIF(
            SCR_URL IS NOT NULL
            AND TRIM(SCR_URL) != ""
            AND NOT REGEXP_CONTAINS(TRIM(SCR_URL), r"^https?://")
        ) AS NON_HTTP_URLS,
        COUNT(DISTINCT SCR_URL) AS DISTINCT_URLS
    FROM {TABLE}
    WHERE {WHERE_SQL}
    """
    display(run_query(url_sql, "url_quality").T)
else:
    print("SCR_URL is not present in the physical table.")

## 18. Cross-field consistency checks

These checks identify exceptions that may indicate source-data issues or misunderstood business rules.

They are **not automatic data-quality failures**; exceptions need validation with the data owner / SME.

In [ ]:
consistency_sql = f"""
SELECT
    COUNTIF(UPDATED_ON_DTM < CREATED_ON_DTM) AS UPDATE_BEFORE_CREATE,
    COUNTIF(IS_ACTV_IND = 1 AND (STAT_CDE IS NULL OR TRIM(STAT_CDE) = "")) AS ACTIVE_WITHOUT_STATUS,
    COUNTIF(ORIGIN_ALERT_ID IS NOT NULL AND (RECORD_CAT_CDE IS NULL OR TRIM(RECORD_CAT_CDE) = "")) AS ALERT_WITHOUT_CATEGORY,
    COUNTIF((TITLE IS NULL OR TRIM(TITLE) = "") AND SUMM_CONTNT IS NOT NULL AND TRIM(SUMM_CONTNT) != "") AS SUMMARY_WITHOUT_TITLE,
    COUNTIF(SUMM_CONTNT IS NOT NULL AND LENGTH(TRIM(SUMM_CONTNT)) < 20) AS VERY_SHORT_SUMMARIES
FROM {TABLE}
WHERE {WHERE_SQL}
"""

display(run_query(consistency_sql, "consistency").T)

## 19. Controlled sample for manual inspection

Automated profiling should be complemented by human review.

This sample is deliberately limited. For production data, review representative records across jurisdiction, status, category and content-length buckets rather than downloading the full population.

In [ ]:
sample_sql = f"""
SELECT
    RECORD_ID,
    ORIGIN_ALERT_ID,
    JRIS_CDE,
    RGLT_BDY_CDE,
    RECORD_CAT_CDE,
    STAT_CDE,
    IS_ACTV_IND,
    CREATED_ON_DTM,
    UPDATED_ON_DTM,
    TITLE,
    SUMM_CONTNT,
    INTRO,
    SCR_URL
FROM {TABLE}
WHERE {WHERE_SQL}
  AND SUMM_CONTNT IS NOT NULL
  AND TRIM(SUMM_CONTNT) != ""
ORDER BY FARM_FINGERPRINT(CAST(RECORD_ID AS STRING))
LIMIT {SAMPLE_N}
"""

sample_df = run_query(sample_sql, "manual_sample")
print("Sample shape:", sample_df.shape)
display(sample_df.head(20))

## 20. AI-readiness summary

This compact output translates the raw EDA into indicators useful for the first project discussion.

The percentages should be read as **diagnostic indicators**, not acceptance criteria.

In [ ]:
ai_readiness_sql = f"""
SELECT
    COUNT(*) AS TOTAL_RECORDS,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_IDS,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(TITLE IS NOT NULL AND TRIM(TITLE) != "" AND SUMM_CONTNT IS NOT NULL AND TRIM(SUMM_CONTNT) != ""),
        COUNT(*)
    ), 2) AS PCT_TITLE_AND_SUMMARY,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(SUMM_CONTNT IS NOT NULL AND LENGTH(TRIM(SUMM_CONTNT)) >= 500),
        COUNT(*)
    ), 2) AS PCT_SUMMARY_GE_500_CHARS,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(SUMM_CONTNT IS NOT NULL AND LENGTH(TRIM(SUMM_CONTNT)) >= 1000),
        COUNT(*)
    ), 2) AS PCT_SUMMARY_GE_1000_CHARS,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(SCR_URL IS NOT NULL AND REGEXP_CONTAINS(TRIM(SCR_URL), r"^https?://")),
        COUNT(*)
    ), 2) AS PCT_HTTP_SOURCE_URL,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(JRIS_CDE IS NOT NULL AND TRIM(JRIS_CDE) != ""),
        COUNT(*)
    ), 2) AS PCT_JURISDICTION_AVAILABLE,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(RGLT_BDY_CDE IS NOT NULL AND TRIM(RGLT_BDY_CDE) != ""),
        COUNT(*)
    ), 2) AS PCT_REGULATOR_AVAILABLE
FROM {TABLE}
WHERE {WHERE_SQL}
"""

display(run_query(ai_readiness_sql, "ai_readiness").T)

# 21. Additional expert-level EDA recommended for the project

The baseline above is enough for the **first RAPID2 data-understanding cycle**. The following analyses should be added because they directly support AI solution decisions.

### 1. Referential integrity

Measure unmatched keys for:
- jurisdiction → jurisdiction reference;
- regulator → regulator reference;
- risk steward area → reference;
- alert ID → alert source;
- theme → theme reference;
- risk taxonomy → taxonomy reference.

### 2. Join cardinality

For every join used by the solution, quantify:
- 1:1;
- 1:many;
- many:1;
- unmatched left rows;
- duplicate right-side keys;
- row multiplication after the join.

### 3. Historical change analysis

If the source preserves history/version information, measure:
- records changed per month;
- field-level changes;
- status transitions;
- taxonomy changes;
- summary changes;
- time between regulatory publication and RAPID2 availability.

### 4. Near-duplicate / boilerplate analysis

Exact duplicates are only the first layer. Consider normalised text hashes, MinHash/LSH and later semantic similarity.

### 5. Coverage matrix

Build jurisdiction × regulator × category × status matrices to identify missing or unusually sparse combinations.

### 6. AI retrieval suitability

Test real business questions against the data model, for example:
- What regulations affect jurisdiction X?
- What changed in the last N days?
- Which regulator issued this record?
- Which records relate to risk taxonomy X?
- Can an answer be traced to a stable record and source URL?

### 7. Baseline drift monitoring

Persist this EDA output as a baseline and compare future data for:
- volume drift;
- null-rate drift;
- category drift;
- jurisdiction drift;
- taxonomy drift;
- text-length drift.

### 8. Security / sensitivity screening

Before AI ingestion, explicitly assess whether regulatory text contains confidential, personal or otherwise restricted information and whether downstream AI components are permitted to process it.

# 22. What this notebook should give the project team

At the end of this run, we should be able to answer the initial RAPID2 questions with evidence rather than assumptions:

| Area | Initial question |
|---|---|
| Scope | What population are we actually analysing? |
| Volume | How large is it? |
| Grain | What does one RAPID2 record represent? |
| Freshness | How current is the data? |
| Coverage | Which jurisdictions / regulators / categories dominate? |
| Quality | Which fields are sparse or inconsistent? |
| Uniqueness | Is `RECORD_ID` reliable? |
| Content | Is there sufficient regulatory text for AI? |
| Metadata | Are status / regulator / taxonomy fields populated? |
| Relationships | Can records be safely enriched without row multiplication? |
| Traceability | Can AI answers point back to a source record / URL? |
| Change | Can we identify regulatory updates over time? |
| AI suitability | Which initial AI use cases are realistically supported? |

**Do not certify the source as “good” or “bad” from this notebook alone.** Use the exceptions and distributions to drive targeted questions with the RAPID2 data owner, Reg Management SMEs, IT and the strategic-solution team.

---

### Next EDA

The next notebook should cover **RegMap tactical extracts**, with much heavier emphasis on **entity relationships, primary/business keys, join cardinality, orphan records and cross-extract reconciliation**, because those seven extracts form a connected regulatory-management model.